# Feature Families + Binary Target Engineering Matrix

This notebook checks whether Quant Warehouse can combine reusable feature families with binary target-engineering labels. It focuses on joinability, target sparsity, actual-event positive rates, and family-level signal separation rather than Rank IC.

Targets covered here:

- FMP event pairs: congress buy/sell, insider buy/sell, analyst upgrades/downgrades, price target raises/cuts, earnings beats/misses.
- Oracle trade entry labels from stored prices: yearly top-1 through top-12 long/short entries using the batched optimal-trade solver with `YE: [1..12]`. Execution prices match `optimal_trader`: buy at high, sell at low, short at low, cover at high.

Important rate convention:

- `rows` is full symbol/date coverage for the target column.
- `rate_rows` is the actual paired-event denominator.
- `positive_rate` for paired labels is computed on actual paired rows, e.g. congress buy / (congress buy + congress sell), not buy / all symbol-date rows.

The goal is to answer: for each feature family, which target families have enough actual joined rows to be worth deeper modeling?

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'quant_warehouse').exists():
    REPO_ROOT = next(parent for parent in Path.cwd().parents if (parent / 'quant_warehouse').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    combine_target_panels,
    evaluate_feature_target_matrix,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
    summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

In [2]:
RUN_STARTED = perf_counter()

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider='fmp',
    market_cap_min=10_000_000_000,
    start_date='2018-01-01',
    horizons=(20, 60),
    min_observations=120,
    max_features_per_family=50,
)

TARGET_CONFIG = BinaryTargetConfig(
    provider='fmp',
    start_date=FEATURE_CONFIG.start_date,
    end_date=FEATURE_CONFIG.end_date,
    event_families=(
        'congress',
        'insider',
        'analyst_rating',
        'price_target',
        'earnings',
    ),
    oracle_trade_k_by_frequency={'YE': tuple(range(1, 13))},
    oracle_trade_min_profit_pct=0.01,
    oracle_trade_long_entry_price_col='high',  # buy execution
    oracle_trade_long_exit_price_col='low',    # sell execution
    oracle_trade_short_entry_price_col='low',  # short execution
    oracle_trade_short_exit_price_col='high',  # cover execution
)

FEATURE_CONFIG, TARGET_CONFIG

WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(
    config=WAREHOUSE.config,
    backend=WAREHOUSE.backend,
    catalog=WAREHOUSE.catalog,
    fundamentals=WAREHOUSE.fundamentals,
    equity_calendar=WAREHOUSE.equity_calendar,
)

## Universe and Feature Families

The notebook starts with FMP equities screened through the OpenBB/FMP screener route, then keeps symbols with the locally stored sections required for the fundamental feature families.

In [3]:
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)

print(f'Universe source: {universe_source}')
print(f'Eligible symbols: {len(symbols)}')
print(symbols[:50])

eligibility.head(20)

Universe source: openbb:fmp
Eligible symbols: 740
('A', 'AA', 'AAL', 'AAOI', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEIS', 'AEP', 'AES', 'AFG', 'AFL', 'AFRM', 'AGNC', 'AGX', 'AHR', 'AIG', 'AIT', 'AIZ', 'AIZN', 'AJG', 'AKAM', 'ALAB', 'ALB', 'ALGM', 'ALGN', 'ALL', 'ALLY', 'ALNY', 'AM', 'AMAT', 'AMD', 'AME', 'AMGN', 'AMH', 'AMKR', 'AMP', 'AMT', 'AMZN', 'ANET', 'APA', 'APC')


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4785585180000
1,GOOGL,True,ok,4368788098535
2,GOOG,True,ok,4349493623926
3,AAPL,True,ok,4323663859280
4,MSFT,True,ok,2854597080400
5,AMZN,True,ok,2599991070000
6,SPCX,True,ok,2059971841733
7,AVGO,True,ok,1757164597200
8,TSLA,True,ok,1597307716000
9,META,True,ok,1555825021126


In [4]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)

print({
    'screened_symbols': len(symbols),
    'raw_feature_panel_rows': len(raw_feature_panel),
    'raw_features': raw_feature_metadata['feature'].nunique(),
    **feature_timings,
})

feature_diagnostics.sort_values(['status', 'rows'], ascending=[True, False]).head(20)

{'screened_symbols': 740, 'raw_feature_panel_rows': 1456399, 'raw_features': 418, 'raw_panel_build_seconds': 63.73798150708899}


,symbol,status,rows,features,seconds
0,A,ok,2130,376,0.236466
1,AA,ok,2130,376,0.068307
2,AAL,ok,2130,376,0.064033
3,AAOI,ok,2130,376,0.071482
4,AAPL,ok,2130,376,0.068573
5,ABBV,ok,2130,376,0.068842
7,ABT,ok,2130,376,0.074820
8,ACN,ok,2130,376,0.068046
9,ADBE,ok,2130,376,0.078378
10,ADI,ok,2130,376,0.066957


## Binary Event Targets

Event targets are loaded without remote refresh. Cached normalized event-pair data is used when present; missing families are built from already stored FMP historical sections.

In [5]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    TARGET_CONFIG,
    event_store=EVENT_STORE,
    include_historical=True,
)

event_symbols = tuple(
    event_diagnostics
    .loc[event_diagnostics['combined_rows'].gt(0), 'symbol']
    .sort_values()
    .tolist()
)
feature_panel = raw_feature_panel.loc[raw_feature_panel['symbol'].isin(event_symbols)].copy()
feature_metadata = raw_feature_metadata.copy()
selected_features, capped_feature_metadata, feature_quality = cap_features_by_quality(
    feature_panel,
    feature_metadata,
    max_features=FEATURE_CONFIG.max_features_per_family,
)
feature_inventory = (
    capped_feature_metadata
    .groupby(['source', 'family'])
    .agg(feature_count=('feature', 'nunique'))
    .reset_index()
    .sort_values(['source', 'family'])
)

event_target_panel, event_target_metadata = build_event_target_panel(
    feature_panel[['symbol', 'date']],
    events,
    TARGET_CONFIG,
)

print({
    'screened_symbols': len(symbols),
    'event_symbols': len(event_symbols),
    'event_rows': len(events),
    'feature_panel_rows_after_event_symbol_filter': len(feature_panel),
    'capped_features': len(selected_features),
    'event_target_rows': len(event_target_panel),
    'event_target_columns': len(event_target_metadata),
    'event_load_seconds': round(event_load_seconds, 3),
})

display(feature_inventory)
event_diagnostics.sort_values('combined_rows', ascending=False).head(20)

{'screened_symbols': 740, 'event_symbols': 731, 'event_rows': 26238, 'feature_panel_rows_after_event_symbol_filter': 1447676, 'capped_features': 390, 'event_target_rows': 1447676, 'event_target_columns': 10, 'event_load_seconds': 15.468}


,source,family,feature_count
0,financetoolkit,ft_growth_balance,50
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
3,financetoolkit,ft_ratios_efficiency,5
4,financetoolkit,ft_ratios_liquidity,7
5,financetoolkit,ft_ratios_profitability,14
6,financetoolkit,ft_ratios_solvency,15
7,financetoolkit,ft_ratios_valuation,24
8,fmp,fmp_balance_mcap,50
9,fmp,fmp_cash_mcap,50


,symbol,cached_rows,historical_rows,combined_rows,event_families
443,MSFT,34,945,979,"(analyst_rating, congress, earnings, insider, ..."
4,AAPL,32,820,852,"(analyst_rating, congress, earnings, insider, ..."
471,NVDA,25,799,824,"(analyst_rating, congress, earnings, insider, ..."
46,AMZN,34,719,753,"(analyst_rating, congress, earnings, insider, ..."
305,GOOGL,34,581,615,"(analyst_rating, congress, earnings, insider, ..."
650,TSLA,32,456,488,"(analyst_rating, congress, earnings, insider, ..."
420,META,34,437,471,"(analyst_rating, congress, earnings, insider, ..."
181,CW,0,34,34,"(earnings,)"
317,HD,0,34,34,"(earnings,)"
150,COF,0,34,34,"(earnings,)"


In [6]:
oracle_target_panel, oracle_target_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols,
    TARGET_CONFIG,
    warehouse=WAREHOUSE,
)

target_panel = combine_target_panels(event_target_panel, oracle_target_panel)
target_metadata = pd.concat([event_target_metadata, oracle_target_metadata], ignore_index=True)
target_summary = summarize_binary_targets(target_panel, target_metadata)

print({
    'oracle_target_rows': len(oracle_target_panel),
    'oracle_target_columns': len(oracle_target_metadata),
    'oracle_k': TARGET_CONFIG.oracle_trade_k_by_frequency,
    'oracle_seconds': round(oracle_seconds, 3),
    'combined_target_rows': len(target_panel),
    'combined_target_columns': target_metadata['target'].nunique(),
})

display(target_summary)

rate_audit = target_summary[[
    'target', 'target_family', 'rows', 'rate_rows', 'positive_rows', 'positive_rate', 'positive_symbols'
]].copy()
rate_audit['positive_rate_pct'] = (rate_audit['positive_rate'] * 100).round(2)
display(rate_audit.sort_values(['target_family', 'rate_rows', 'positive_rows'], ascending=[True, False, False]))


{'oracle_target_rows': 1447676, 'oracle_target_columns': 36, 'oracle_k': {'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}, 'oracle_seconds': 50.219, 'combined_target_rows': 1447676, 'combined_target_columns': 46}


,target,target_family,rows,rate_rows,positive_rows,positive_rate,positive_symbols,min_positive_date,max_positive_date
0,target_event_on__earnings_beat,event,1447676,21437,16852,0.786117,726,2018-01-04,2026-06-04
1,target_event_on__earnings_miss,event,1447676,21437,4585,0.213883,665,2018-01-09,2026-05-28
2,target_event_on__congress_buy,event,1447676,2496,1374,0.550481,7,2018-01-25,2026-06-05
3,target_event_on__congress_sell,event,1447676,2496,1341,0.537260,7,2018-01-04,2026-06-16
4,target_event_on__insider_sell,event,1447676,538,534,0.992565,7,2018-08-08,2026-06-18
5,target_event_on__analyst_upgrade,event,1447676,268,214,0.798507,7,2024-04-04,2026-06-09
6,target_event_on__price_target_raise,event,1447676,245,202,0.824490,7,2024-01-11,2026-06-22
7,target_event_on__analyst_downgrade,event,1447676,268,68,0.253731,7,2024-04-03,2026-06-22
8,target_event_on__price_target_cut,event,1447676,245,60,0.244898,7,2024-01-02,2026-06-09
9,target_event_on__insider_buy,event,1447676,538,4,0.007435,2,2021-03-10,2026-02-18


,target,target_family,rows,rate_rows,positive_rows,positive_rate,positive_symbols,positive_rate_pct
0,target_event_on__earnings_beat,event,1447676,21437,16852,0.786117,726,78.61
1,target_event_on__earnings_miss,event,1447676,21437,4585,0.213883,665,21.39
2,target_event_on__congress_buy,event,1447676,2496,1374,0.550481,7,55.05
3,target_event_on__congress_sell,event,1447676,2496,1341,0.537260,7,53.73
4,target_event_on__insider_sell,event,1447676,538,534,0.992565,7,99.26
9,target_event_on__insider_buy,event,1447676,538,4,0.007435,2,0.74
5,target_event_on__analyst_upgrade,event,1447676,268,214,0.798507,7,79.85
7,target_event_on__analyst_downgrade,event,1447676,268,68,0.253731,7,25.37
6,target_event_on__price_target_raise,event,1447676,245,202,0.824490,7,82.45
8,target_event_on__price_target_cut,event,1447676,245,60,0.244898,7,24.49


## Feature Family x Target Matrix

This matrix asks whether a model row can be formed for each pair. `mean_abs_smd` is a fast family-level separation check: larger values mean the positive and negative target rows look more different on average. It is not a full trading evaluation.

In [7]:
matrix, training_panel = evaluate_feature_target_matrix(
    feature_panel,
    capped_feature_metadata,
    target_panel,
    target_metadata,
    min_rows=120,
    min_positive_rows=10,
    min_feature_coverage=0.5,
)

usable = matrix.query("status == 'usable'").copy()

print({
    'matrix_rows': len(matrix),
    'usable_pairs': len(usable),
    'training_panel_rows': len(training_panel),
    'elapsed_seconds': round(perf_counter() - RUN_STARTED, 3),
})

display(usable.head(50))

usable_rate_audit = usable[[
    'source', 'feature_family', 'target_family', 'target', 'rows', 'positive_rows', 'positive_rate', 'mean_abs_smd', 'status'
]].copy()
usable_rate_audit['positive_rate_pct'] = (usable_rate_audit['positive_rate'] * 100).round(2)
display(usable_rate_audit.sort_values(['target_family', 'target', 'mean_abs_smd'], ascending=[True, True, False]).head(80))


{'matrix_rows': 690, 'usable_pairs': 675, 'training_panel_rows': 1447676, 'elapsed_seconds': 238.89}


,source,feature_family,target_family,target,feature_count,rows,positive_rows,positive_rate,mean_feature_coverage,mean_abs_smd,status
15,financetoolkit,ft_growth_income,oracle_trade,target_oracle_trade_entry__YE_k12_any,38,132285,132285,1.0,0.763158,NaN,usable
16,financetoolkit,ft_growth_cash,oracle_trade,target_oracle_trade_entry__YE_k12_any,50,132282,132282,1.0,0.740000,NaN,usable
17,financetoolkit,ft_ratios_efficiency,oracle_trade,target_oracle_trade_entry__YE_k12_any,5,132279,132279,1.0,1.000000,NaN,usable
18,financetoolkit,ft_ratios_liquidity,oracle_trade,target_oracle_trade_entry__YE_k12_any,7,132279,132279,1.0,1.000000,NaN,usable
19,financetoolkit,ft_ratios_profitability,oracle_trade,target_oracle_trade_entry__YE_k12_any,14,132279,132279,1.0,1.000000,NaN,usable
20,financetoolkit,ft_ratios_solvency,oracle_trade,target_oracle_trade_entry__YE_k12_any,15,132279,132279,1.0,1.000000,NaN,usable
21,financetoolkit,ft_ratios_valuation,oracle_trade,target_oracle_trade_entry__YE_k12_any,24,132279,132279,1.0,0.991979,NaN,usable
22,fmp,fmp_daily_ev_multiple,oracle_trade,target_oracle_trade_entry__YE_k12_any,7,132274,132274,1.0,0.998126,NaN,usable
23,fmp,fmp_daily_ev_yield,oracle_trade,target_oracle_trade_entry__YE_k12_any,7,132269,132269,1.0,0.999984,NaN,usable
24,financetoolkit,ft_growth_balance,oracle_trade,target_oracle_trade_entry__YE_k12_any,50,132260,132260,1.0,0.988427,NaN,usable


,source,feature_family,target_family,target,rows,positive_rows,positive_rate,mean_abs_smd,status,positive_rate_pct
660,financetoolkit,ft_ratios_efficiency,event,target_event_on__analyst_downgrade,268,68,0.253731,0.484126,usable,25.37
661,fmp,fmp_balance_mcap,event,target_event_on__analyst_downgrade,268,68,0.253731,0.378586,usable,25.37
662,financetoolkit,ft_ratios_valuation,event,target_event_on__analyst_downgrade,268,68,0.253731,0.285837,usable,25.37
663,financetoolkit,ft_growth_income,event,target_event_on__analyst_downgrade,268,68,0.253731,0.285609,usable,25.37
664,financetoolkit,ft_ratios_solvency,event,target_event_on__analyst_downgrade,268,68,0.253731,0.277958,usable,25.37
...,...,...,...,...,...,...,...,...,...,...
580,fmp,fmp_daily_mcap_yield,event,target_event_on__earnings_miss,21375,4571,0.213848,0.052778,usable,21.38
581,fmp,fmp_income_mcap,event,target_event_on__earnings_miss,21375,4571,0.213848,0.051753,usable,21.38
584,fmp,fmp_balance_mcap,event,target_event_on__earnings_miss,21373,4570,0.213821,0.050912,usable,21.38
570,financetoolkit,ft_ratios_profitability,event,target_event_on__earnings_miss,21410,4579,0.213872,0.046896,usable,21.39


In [8]:
coverage_pivot = (
    matrix
    .pivot_table(
        index=['source', 'feature_family'],
        columns='target_family',
        values='target',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

positive_pivot = (
    usable
    .pivot_table(
        index=['source', 'feature_family'],
        columns='target_family',
        values='positive_rows',
        aggfunc='max',
        fill_value=0,
    )
    .reset_index()
)

coverage_pivot

target_family,source,feature_family,event,oracle_trade
0,financetoolkit,ft_growth_balance,10,36
1,financetoolkit,ft_growth_cash,10,36
2,financetoolkit,ft_growth_income,10,36
3,financetoolkit,ft_ratios_efficiency,10,36
4,financetoolkit,ft_ratios_liquidity,10,36
5,financetoolkit,ft_ratios_profitability,10,36
6,financetoolkit,ft_ratios_solvency,10,36
7,financetoolkit,ft_ratios_valuation,10,36
8,fmp,fmp_balance_mcap,10,36
9,fmp,fmp_cash_mcap,10,36


In [9]:
best_by_target = (
    usable
    .sort_values(['target', 'mean_abs_smd', 'positive_rows'], ascending=[True, False, False])
    .groupby('target')
    .head(3)
    .reset_index(drop=True)
)

best_by_target

,source,feature_family,target_family,target,feature_count,rows,positive_rows,positive_rate,mean_feature_coverage,mean_abs_smd,status
0,financetoolkit,ft_ratios_efficiency,event,target_event_on__analyst_downgrade,5,268,68,0.253731,1.000000,0.484126,usable
1,fmp,fmp_balance_mcap,event,target_event_on__analyst_downgrade,50,268,68,0.253731,1.000000,0.378586,usable
2,financetoolkit,ft_ratios_valuation,event,target_event_on__analyst_downgrade,24,268,68,0.253731,1.000000,0.285837,usable
3,financetoolkit,ft_ratios_efficiency,event,target_event_on__analyst_upgrade,5,268,214,0.798507,1.000000,0.415605,usable
4,fmp,fmp_balance_mcap,event,target_event_on__analyst_upgrade,50,268,214,0.798507,1.000000,0.364155,usable
...,...,...,...,...,...,...,...,...,...,...,...
130,fmp,fmp_cash_mcap,oracle_trade,target_oracle_trade_entry__YE_k9_long,50,105669,53195,0.503412,0.779709,0.049074,usable
131,fmp,fmp_balance_mcap,oracle_trade,target_oracle_trade_entry__YE_k9_long,50,105620,53170,0.503408,0.999989,0.012351,usable
132,fmp,fmp_income_mcap,oracle_trade,target_oracle_trade_entry__YE_k9_short,45,105673,52476,0.496589,0.688674,0.144711,usable
133,fmp,fmp_cash_mcap,oracle_trade,target_oracle_trade_entry__YE_k9_short,50,105669,52474,0.496588,0.779709,0.049074,usable


In [10]:
sparse_targets = target_summary.query('positive_rows < 10').copy()
usable_targets = usable['target'].nunique() if not usable.empty else 0
usable_feature_families = usable['feature_family'].nunique() if not usable.empty else 0
best_pairs = usable.sort_values('mean_abs_smd', ascending=False).head(10) if not usable.empty else usable

lines = [
    '## Written Analysis',
    '',
    f'- Universe: {len(symbols)} screened FMP symbols with market cap >= ${FEATURE_CONFIG.market_cap_min:,.0f}; {len(event_symbols)} symbols had at least one loaded event pair and were used in the target matrix.',
    f'- Feature side: {capped_feature_metadata["family"].nunique()} capped feature families and {len(selected_features)} selected feature columns.',
    f'- Target side: {target_metadata["target"].nunique()} binary target columns across same-day event and oracle-trade families; oracle is configured as {TARGET_CONFIG.oracle_trade_k_by_frequency}.',
    f'- Joinability: {len(usable)} feature-family/target pairs are usable under the current thresholds, covering {usable_targets} target columns and {usable_feature_families} feature families. For paired event/oracle labels, matrix rows are actual paired rows, not all no-event dates.',
]

if not sparse_targets.empty:
    lines.append(f'- Sparse targets: {len(sparse_targets)} target columns have fewer than 10 positive rows and should not be trusted for modeling yet.')

if 'rate_rows' in target_summary.columns:
    event_rate_preview = target_summary.sort_values(['target_family', 'rate_rows'], ascending=[True, False]).head(8)
    lines.append('')
    lines.append('Largest actual-event denominators:')
    for row in event_rate_preview[['target', 'rate_rows', 'positive_rows', 'positive_rate']].to_dict('records'):
        rate_pct = row['positive_rate'] * 100 if pd.notna(row['positive_rate']) else float('nan')
        lines.append(f'- {row["target"]}: {int(row["positive_rows"])} positive / {int(row["rate_rows"])} paired rows ({rate_pct:.2f}%).')

if not best_pairs.empty:
    lines.append('')
    lines.append('Top family/target combinations by fast separation score:')
    for row in best_pairs[['source', 'feature_family', 'target', 'positive_rows', 'mean_abs_smd']].to_dict('records'):
        lines.append(
            f'- {row["source"]}.{row["feature_family"]} -> {row["target"]}: '
            f'{int(row["positive_rows"])} positives, mean_abs_smd={row["mean_abs_smd"]:.4f}'
        )

lines.extend([
    '',
    'Interpretation:',
    '',
    '- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.',
    '- Oracle-trade labels are sparse yearly top-1 through top-12 entry labels from the batched optimal-trade solver; this keeps dataset size manageable while preserving the buy/sell oracle task shape.',
    '- Congress, insider, analyst, price-target, and earnings labels are same-day sparse events. Their positive rates now use only actual paired event rows, which is the right denominator for buy/sell or upgrade/downgrade questions.',
    '- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.',
])

display(Markdown('\n'.join(lines)))

## Written Analysis

- Universe: 740 screened FMP symbols with market cap >= $10,000,000,000; 731 symbols had at least one loaded event pair and were used in the target matrix.
- Feature side: 15 capped feature families and 390 selected feature columns.
- Target side: 46 binary target columns across same-day event and oracle-trade families; oracle is limited to {'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}.
- Joinability: 675 feature-family/target pairs are usable under the current thresholds, covering 45 target columns and 15 feature families. For paired event/oracle labels, matrix rows are actual paired rows, not all no-event dates.
- Sparse targets: 1 target columns have fewer than 10 positive rows and should not be trusted for modeling yet.

Largest actual-event denominators:
- target_event_on__earnings_beat: 16852 positive / 21437 paired rows (78.61%).
- target_event_on__earnings_miss: 4585 positive / 21437 paired rows (21.39%).
- target_event_on__congress_buy: 1374 positive / 2496 paired rows (55.05%).
- target_event_on__congress_sell: 1341 positive / 2496 paired rows (53.73%).
- target_event_on__insider_sell: 534 positive / 538 paired rows (99.26%).
- target_event_on__insider_buy: 4 positive / 538 paired rows (0.74%).
- target_event_on__analyst_upgrade: 214 positive / 268 paired rows (79.85%).
- target_event_on__analyst_downgrade: 68 positive / 268 paired rows (25.37%).

Top family/target combinations by fast separation score:
- financetoolkit.ft_ratios_efficiency -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.7885
- fmp.fmp_daily_mcap_multiple -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.6343
- financetoolkit.ft_ratios_profitability -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.6048
- fmp.fmp_daily_ev_multiple -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.5730
- financetoolkit.ft_ratios_liquidity -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.5198
- financetoolkit.ft_ratios_efficiency -> target_event_on__analyst_downgrade: 68 positives, mean_abs_smd=0.4841
- financetoolkit.ft_ratios_solvency -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4755
- fmp.fmp_daily_ev_yield -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4711
- fmp.fmp_cash_mcap -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4547
- financetoolkit.ft_growth_balance -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4545

Interpretation:

- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.
- Oracle-trade labels are sparse yearly top-1 through top-12 entry labels from the batched optimal-trade solver; this keeps dataset size manageable while preserving the buy/sell oracle task shape.
- Congress, insider, analyst, price-target, and earnings labels are same-day sparse events. Their positive rates now use only actual paired event rows, which is the right denominator for buy/sell or upgrade/downgrade questions.
- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.

## Written Analysis

- Universe: 740 screened FMP symbols with market cap >= $10,000,000,000; 731 symbols had at least one loaded event pair and were used in the target matrix.
- Feature side: 15 capped feature families and 390 selected feature columns.
- Target side: 46 binary target columns across same-day event and oracle-trade families; oracle is configured as {'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}.
- Joinability: 675 feature-family/target pairs are usable under the current thresholds, covering 45 target columns and 15 feature families. For paired event/oracle labels, matrix rows are actual paired rows, not all no-event dates.
- Sparse targets: 1 target columns have fewer than 10 positive rows and should not be trusted for modeling yet.

Largest actual-event denominators:
- target_event_on__earnings_beat: 16852 positive / 21437 paired rows (78.61%).
- target_event_on__earnings_miss: 4585 positive / 21437 paired rows (21.39%).
- target_event_on__congress_buy: 1374 positive / 2496 paired rows (55.05%).
- target_event_on__congress_sell: 1341 positive / 2496 paired rows (53.73%).
- target_event_on__insider_sell: 534 positive / 538 paired rows (99.26%).
- target_event_on__insider_buy: 4 positive / 538 paired rows (0.74%).
- target_event_on__analyst_upgrade: 214 positive / 268 paired rows (79.85%).
- target_event_on__analyst_downgrade: 68 positive / 268 paired rows (25.37%).

Top family/target combinations by fast separation score:
- financetoolkit.ft_ratios_efficiency -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.7885
- fmp.fmp_daily_mcap_multiple -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.6343
- financetoolkit.ft_ratios_profitability -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.6048
- fmp.fmp_daily_ev_multiple -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.5730
- financetoolkit.ft_ratios_liquidity -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.5198
- financetoolkit.ft_ratios_efficiency -> target_event_on__analyst_downgrade: 68 positives, mean_abs_smd=0.4841
- financetoolkit.ft_ratios_solvency -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4755
- fmp.fmp_daily_ev_yield -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4711
- fmp.fmp_cash_mcap -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4547
- financetoolkit.ft_growth_balance -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.4545

Interpretation:

- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.
- Oracle-trade labels are sparse yearly top-1 through top-12 entry labels from the batched optimal-trade solver; this keeps dataset size manageable while preserving the buy/sell oracle task shape.
- Congress, insider, analyst, price-target, and earnings labels are same-day sparse events. Their positive rates now use only actual paired event rows, which is the right denominator for buy/sell or upgrade/downgrade questions.
- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.